# Project1 — Huấn luyện Model 1 trên Google Colab
Mặc định chỉ thiết lập và kiểm tra môi trường, chưa huấn luyện hoặc tải mô hình.
Model 1 có hai lựa chọn: PhoBERT CLS + LightGBM trên CPU hoặc PhoBERT-base-v2 + LoRA trên GPU T4. Bật `RUN_LORA` để huấn luyện LoRA sau khi nhận đủ dữ liệu.
Xem hướng dẫn tại `docs/COLAB_SETUP.md`. Không cần cài MCP để sử dụng notebook.


## 1. Cấu hình và kết nối Google Drive
Nếu kho mã nguồn ở chế độ riêng tư hoặc có thay đổi trên máy chưa được đẩy lên GitHub, hãy sao chép mã nguồn lên Drive và chọn `SOURCE_MODE = "drive"`. Không sao chép thông tin đăng nhập, khóa bí mật hoặc thư mục môi trường ảo.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys
import shutil

SOURCE_MODE = "github"  # "github" or "drive"
REPO_URL = "https://github.com/Drista4real/Project1.git"
REPO_REF = "develop"  # Use a commit SHA for reproducibility
DRIVE_SOURCE = Path("/content/drive/MyDrive/Project1/source")
INSTALL_LORA = False
RUN_CLASSIFIER = False  # Optional CPU LightGBM baseline
RUN_LORA = False  # Model 1: PhoBERT-base-v2 + LoRA on CUDA
LORA_MODEL = "vinai/phobert-base-v2"
LORA_REVISION = "main"  # Or pin a Hugging Face commit SHA
LORA_EPOCHS = 5
LORA_BATCH_SIZE = 8
LORA_GRADIENT_ACCUMULATION = 4
LORA_MAX_LENGTH = 128
LORA_LEARNING_RATE = 2e-4
RUN_FORECASTER = False
DOWNLOAD_ZIP = False
DATASET_DIR = None  # Example: "/content/drive/MyDrive/Project1/datasets/bundle_v2"
FORECAST_CSV = None  # None uses data/processed/balance_fixed.csv

from google.colab import drive

drive.mount("/content/drive")
WORK_ROOT = Path("/content/project1_colab")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DRIVE = Path("/content/drive/MyDrive/Project1")
from uuid import uuid4
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
RUN_DIR = PROJECT_DRIVE / "colab_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CACHE_DIR = PROJECT_DRIVE / "model_cache/phobert"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Run artifacts:", RUN_DIR)
LORA_CACHE_DIR = PROJECT_DRIVE / "model_cache/phobert-lora"
LORA_CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Nạp mã nguồn
Mỗi lần chạy sử dụng một bản sao mã nguồn mới trên ổ đĩa tạm của Colab. Nhánh hoặc phiên bản GitHub được chọn phải chứa các tệp khai báo thư viện mới; nếu chưa có, hãy dùng bản mã nguồn đã cập nhật trên Drive.


In [ ]:
def run_git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"Git failed (exit {result.returncode}):\n{result.stderr.strip()}")
    return result.stdout.strip()


REPO_DIR = WORK_ROOT / RUN_ID
if REPO_DIR.exists():
    raise FileExistsError(
        f"Source directory already exists: {REPO_DIR}. "
        "Run the configuration cell again to create a new RUN_ID, then rerun this cell."
    )
if SOURCE_MODE == "github":
    run_git("clone", "--no-checkout", REPO_URL, str(REPO_DIR))
    # Non-default branches exist only as remote-tracking refs after clone.
    remote_ref = f"refs/remotes/origin/{REPO_REF}"
    is_branch = subprocess.run(
        ["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", remote_ref]
    ).returncode == 0
    selected_ref = remote_ref if is_branch else REPO_REF
    try:
        revision = run_git("-C", str(REPO_DIR), "rev-parse", "--verify",
                           "--end-of-options", f"{selected_ref}^{{commit}}")
    except RuntimeError as exc:
        raise RuntimeError(
            f"Cannot resolve REPO_REF={REPO_REF!r} as a branch, tag or commit. "
            "Check the ref on GitHub, then rerun the configuration cell and this cell.\n"
            f"{exc}"
        ) from exc
    run_git("-C", str(REPO_DIR), "checkout", "--detach", revision)
elif SOURCE_MODE == "drive":
    if not (DRIVE_SOURCE / "backend/scripts/train_classifier.py").is_file():
        raise FileNotFoundError(f"Project source not found at {DRIVE_SOURCE}")
    shutil.copytree(DRIVE_SOURCE, REPO_DIR, ignore=shutil.ignore_patterns(
        ".git", ".venv*", "venv", "node_modules", "__pycache__", ".env", ".env.*",
        "weights", "frontend", ".dart_tool", "build"))
else:
    raise ValueError("SOURCE_MODE must be github or drive")
for name in ("backend/requirements-colab.txt", "backend/scripts/prepare_classifier_data.py"):
    if not (REPO_DIR / name).is_file():
        raise FileNotFoundError(f"Missing {name}; use updated project source")
if SOURCE_MODE == "github":
    commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
else:
    commit = "Drive source snapshot; no verified Git commit"
(RUN_DIR / "source-commit.txt").write_text(commit + "\n", encoding="utf-8")
print("Source:", REPO_DIR, "\nRevision:", commit)


## 3. Cài đặt thư viện học máy
Quá trình cài đặt có thể mất vài phút. Khi bật `RUN_LORA`, notebook tự cài các thư viện LoRA kể cả khi `INSTALL_LORA = False`. Chỉ đặt `INSTALL_LORA = True` nếu muốn kiểm tra thư viện trước khi có dữ liệu.
Các bước học máy chạy trong tiến trình Python mới để nhận đúng phiên bản thư viện vừa cài.


In [ ]:
USE_LORA = INSTALL_LORA or RUN_LORA
requirements = "requirements-colab-lora.txt" if USE_LORA else "requirements-colab.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(REPO_DIR / "backend" / requirements)], check=True)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / "pip-freeze.txt").write_text(freeze, encoding="utf-8")


## 4. Kiểm tra môi trường
Bước này không tải mô hình. Nếu có GPU, chương trình sẽ kiểm tra bằng một phép toán CUDA nhỏ.
Để huấn luyện LoRA, chọn **Runtime → Change runtime type → T4 GPU**. Khi bật `RUN_LORA`, notebook sẽ dừng nếu CUDA không sẵn sàng. Mô hình cơ sở LightGBM vẫn chạy trên CPU.


In [ ]:
def run_python(source, *args):
    from textwrap import dedent
    return subprocess.run([sys.executable, "-c", dedent(source), *map(str, args)],
                          cwd=REPO_DIR, check=True, env={**os.environ, "PYTHONUTF8": "1"})

run_python("""
import importlib.metadata as meta
import json, platform, sys
from pathlib import Path
import numpy, torch, lightgbm, sklearn, transformers
from underthesea import word_tokenize
assert word_tokenize('\u0103n tr\u01b0a', format='text')
cuda = torch.cuda.is_available()
if sys.argv[3] == "True" and not cuda:
    raise RuntimeError("RUN_LORA requires a GPU: select a T4 runtime and rerun setup")
if cuda:
    assert (torch.ones(2, device='cuda') + 1).sum().item() == 4
packages = ['numpy', 'torch', 'lightgbm', 'scikit-learn', 'transformers', 'underthesea']
if sys.argv[2] == 'True':
    import peft, accelerate, datasets
    packages += ['peft', 'accelerate', 'datasets']
report = {'python': sys.version, 'platform': platform.platform(),
          'cuda_available': cuda, 'torch_cuda': torch.version.cuda,
          'gpu': torch.cuda.get_device_name(0) if cuda else None,
          'versions': {name: meta.version(name) for name in packages}}
Path(sys.argv[1]).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))
""", RUN_DIR / "environment.json", USE_LORA, RUN_LORA)


## 5. Kiểm tra dữ liệu
Đặt `DATASET_DIR` thành đường dẫn gói dữ liệu đã được pipeline dự án chuẩn bị: gồm train/validation/test, taxonomy và manifest. Notebook kiểm tra mã băm, nhãn và tính độc lập giữa các tập trước khi huấn luyện.
Chưa có dữ liệu vẫn có thể thiết lập môi trường, nhưng cần giữ `RUN_LORA = False`. Không dùng bundle v1 đã bị báo lệch mã băm. Dữ liệu số dư chỉ được kiểm tra nếu bật `RUN_FORECASTER`.


In [ ]:
CLASSIFIER_READY = False
if DATASET_DIR is None:
    print("No classifier bundle yet. Keep RUN_CLASSIFIER=False and RUN_LORA=False until data handoff.")
else:
    DATASET_DIR = Path(DATASET_DIR)
    subprocess.run([sys.executable, "backend/scripts/prepare_classifier_data.py",
                    "--verify", str(DATASET_DIR)], cwd=REPO_DIR, check=True)
    CLASSIFIER_READY = True
if RUN_FORECASTER:
    FORECAST_PATH = Path(FORECAST_CSV) if FORECAST_CSV else REPO_DIR / "data/processed/balance_fixed.csv"
    run_python("""
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd() / 'backend'))
    from app.infrastructure.ai.forecaster.features import load_daily_cashflows
    rows = load_daily_cashflows(Path(sys.argv[1]))
    if len(rows) < 10:
        raise ValueError('Forecaster needs at least 10 consecutive days')
    print('Valid fixed-income CSV:', len(rows), 'days')
    """, FORECAST_PATH)


## 6. Huấn luyện Model 1 (tùy chọn)
Để chỉ huấn luyện Model 1 có LoRA: đặt `RUN_LORA = True`, giữ `RUN_CLASSIFIER = False`, `RUN_FORECASTER = False` và điền `DATASET_DIR` trong ô cấu hình, rồi chạy lại các ô từ đầu trên T4. Nhánh GitHub hoặc mã nguồn trên Drive phải chứa script LoRA mới.
Mặc định: 5 epoch, batch 8, tích lũy gradient 4 bước, độ dài tối đa 128 token, FP16 trên GPU. Nếu hết bộ nhớ GPU, giảm `LORA_BATCH_SIZE` xuống 4 hoặc 2.
Pipeline chọn epoch tốt nhất theo Macro F1 trên validation, sau đó nạp lại adapter đã lưu để đánh giá test một lần và thử dự đoán. Không dùng tập test để chọn epoch. Kết quả synthetic chưa chứng minh chất lượng trên dữ liệu thật.
Adapter, lớp phân loại, tokenizer, cấu hình và chỉ số được lưu tại `classifier_lora/`; nhật ký nằm trong `classifier_lora.log`. Bản tốt nhất được lưu sau mỗi epoch có cải thiện. Chưa hỗ trợ tiếp tục chính xác một lượt train bị ngắt.


In [ ]:
def train_logged(script, arguments, log_name):
    command = [sys.executable, "-u", str(REPO_DIR / script), *map(str, arguments)]
    with (RUN_DIR / log_name).open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True,
                              encoding="utf-8", errors="replace",
                              env={**os.environ, "PYTHONUTF8": "1"}) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Training failed (exit={status}); see {log_name}")

if RUN_CLASSIFIER:
    if not CLASSIFIER_READY:
        raise ValueError("A verified DATASET_DIR is required before classifier training")
    # The training script verifies the bundle again before using it.
    train_logged("backend/scripts/train_classifier.py", [
        "--dataset-dir", DATASET_DIR, "--output-dir", RUN_DIR / "classifier",
        "--cache-dir", CACHE_DIR, "--batch-size", 16], "classifier.log")
if RUN_LORA:
    if not CLASSIFIER_READY:
        raise ValueError("A verified DATASET_DIR is required before LoRA training")
    if not (REPO_DIR / "backend/scripts/train_classifier_lora.py").is_file():
        raise FileNotFoundError("Update source to the version containing train_classifier_lora.py")
    train_logged("backend/scripts/train_classifier_lora.py", [
        "--dataset-dir", DATASET_DIR, "--output-dir", RUN_DIR / "classifier_lora",
        "--cache-dir", LORA_CACHE_DIR, "--device", "cuda",
        "--model-name", LORA_MODEL, "--revision", LORA_REVISION,
        "--epochs", LORA_EPOCHS, "--batch-size", LORA_BATCH_SIZE,
        "--gradient-accumulation", LORA_GRADIENT_ACCUMULATION,
        "--max-length", LORA_MAX_LENGTH,
        "--learning-rate", LORA_LEARNING_RATE], "classifier_lora.log")
    subprocess.run([sys.executable, "backend/scripts/predict_classifier_lora.py",
                    "--artifact-dir", str(RUN_DIR / "classifier_lora"),
                    "--cache-dir", str(LORA_CACHE_DIR), "--device", "cuda",
                    "--local-files-only", "--text", "ăn trưa 40k"],
                   cwd=REPO_DIR, check=True, env={**os.environ, "PYTHONUTF8": "1"})
if RUN_FORECASTER:
    train_logged("backend/scripts/train_forecaster.py", [
        "--data", FORECAST_PATH, "--output-dir", RUN_DIR / "forecaster"], "forecaster.log")
if not RUN_CLASSIFIER and not RUN_LORA and not RUN_FORECASTER:
    print("Environment prepared; training has not been requested.")


## 7. Xuất kết quả của lần chạy
Tệp ZIP chứa thông tin môi trường, nhật ký và các tệp mô hình đã lưu nếu có. Adapter LoRA nằm trong `classifier_lora/`, kết quả đánh giá tại `classifier_lora/metrics.json`.
Trọng số nền PhoBERT nằm riêng trong `MyDrive/Project1/model_cache/phobert-lora`; khi dự đoán offline cần cả cache nền và thư mục adapter. LoRA có CLI dự đoán riêng, chưa được nối vào API ứng dụng đang dùng LightGBM.


In [ ]:
archive = shutil.make_archive(str(WORK_ROOT / RUN_ID), "zip", root_dir=RUN_DIR)
saved_archive = PROJECT_DRIVE / "colab_runs" / (RUN_ID + ".zip")
shutil.copy2(archive, saved_archive)
print("Saved:", saved_archive)
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(archive)
